### Attribution
https://github.com/miptgirl/miptgirl_medium/blob/main/dspy_example/nps_topic_modelling.ipynb

In [1]:
import pandas as pd
import tqdm

### Try out DSPy on a simple example

In [2]:

SMALL_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-2.5-flash-lite",
    "gemini/gemini-2.5-flash",
    "gemini/gemini-2.0-flash",
]

REFLECTION_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-3.1-pro-preview",
    "gemini/gemini-2.5-pro",
    "gemini/gemini-1.5-pro",
]
import dspy
llm = dspy.LM(SMALL_MODEL_CANDIDATES[0])
dspy.configure(lm=llm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

In [3]:
simple_model = dspy.Predict("question -> answer: int", cache = False)
simple_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")

Prediction(
    answer=5
)

In [4]:
dspy.inspect_history(n = 1)





[2026-05-05T06:41:23.889188]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
5
[[ ## completed ## ]]







In [5]:
from utils import wrap_text

cot_model = dspy.ChainOfThought("question -> answer: int")
answer = cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")

print(f"Answer: {answer.answer}\n")
print("Reasoning:")
print(wrap_text(answer.reasoning, width=80))

dspy.inspect_history(n = 1)





Answer: 5

Reasoning:
The problem asks for the number of possible combinations when selecting 4 balls
out of 5 distinct balls. This is a combination problem, as the order in which
the balls are selected does not matter. The formula for combinations is given by
C(n, k) = n! / (k! * (n-k)!), where n is the total number of items to choose
from, and k is the number of items to choose. In this case, n = 5 (total number
of balls) and k = 4 (number of balls to select).

C(5, 4) = 5! / (4! * (5-4)!)
C(5, 4) = 5! / (4! * 1!)
C(5, 4) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1)
C(5, 4) = 5 / 1
C(5, 4) = 5

Alternatively, choosing 4 balls out of 5 is the same as *not* choosing 1 ball
out of 5. So, C(5, 4) = C(5, 1) = 5.




[2026-05-05T06:41:36.576778]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]


In [6]:
dspy.inspect_history(n = 1)





[2026-05-05T06:41:36.576778]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
The problem asks for the number of po

In [7]:
dspy.configure(adapter=dspy.JSONAdapter())

In [8]:
response = cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")
print(f"Reasoning:\n {wrap_text(response['reasoning'])}")
print(f"Answer: {response['answer']}")

dspy.inspect_history(n = 1)





Reasoning:
 This is a combination problem, as the order of the balls selected does not
matter. We need to find the number of ways to choose 4 balls from a set of 5
distinct balls. The formula for combinations is C(n, k) = n! / (k! * (n-k)!),
where n is the total number of items to choose from, and k is the number of
items to choose. In this case, n = 5 and k = 4. So, C(5, 4) = 5! / (4! * (5-4)!)
= 5! / (4! * 1!) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1) = 5 / 1 = 5.
Answer: 5




[2026-05-05T06:41:44.173976]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must be a single int value"
}
I

### JSONAdapter

In [9]:
dspy.configure(adapter=dspy.JSONAdapter())
print(cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?"))

Prediction(
    reasoning='This is a combination problem, as the order in which the balls are selected does not matter. We are choosing 4 balls from a set of 5. The formula for combinations is C(n, k) = n! / (k! * (n-k)!), where n is the total number of items to choose from, and k is the number of items to choose. In this case, n=5 and k=4. So, C(5, 4) = 5! / (4! * (5-4)!) = 5! / (4! * 1!) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1) = 120 / (24 * 1) = 120 / 24 = 5. Therefore, there are 5 possible combinations.',
    answer=5
)


In [10]:
dspy.inspect_history(n = 1)





[2026-05-05T06:41:45.707192]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with a JSON object in the following order of fields: `reasoning`, then `answer` (must be formatted as a valid Python int).


Response:

{
  "reasoning": "This is a combination problem, as the order in which the balls 

In [11]:
response = cot_model(question="I have 25 different balls and I randomly select 9. How many possible combinations of the balls I can get?")
print(f"Reasoning:\n {wrap_text(response['reasoning'])}")
print(f"Answer: {response['answer']}")
dspy.inspect_history(n = 1)


Reasoning:
 The problem asks for the number of combinations of selecting 9 balls from a set
of 25 distinct balls. This is a combination problem, as the order of selection
does not matter. The formula for combinations is C(n, k) = n! / (k!(n-k)!),
where n is the total number of items to choose from, and k is the number of
items to choose. In this case, n = 25 and k = 9. Therefore, the number of
combinations is C(25, 9) = 25! / (9!(25-9)!) = 25! / (9!16!). Calculating this
value: 25! / (9! * 16!) = (25 * 24 * 23 * 22 * 21 * 20 * 19 * 18 * 17) / (9 * 8
* 7 * 6 * 5 * 4 * 3 * 2 * 1) = 2042975.
Answer: 2042975




[2026-05-05T06:41:47.139477]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fiel

#### Cross-check

In [12]:
import math
n = 25
k = 9
round(math.factorial(n)/math.factorial(k)/math.factorial(n-k))
2042975

2042975

In [13]:
pip install deno

/Users/aurobindotripathy/prompt-opt-cookbook/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from dspy import PythonInterpreter
def evaluate_math(expr: str) -> str:
    # Executes Python and returns the output as string
    with PythonInterpreter() as interp:
        return interp(expr)

react_model = dspy.ReAct(
    signature="question -> answer: int",  # expects an int answer
    tools=[evaluate_math]
)
response = react_model(question="I have 25 different balls and I randomly select 9. How many possible combinations of the balls I can get?")
print(response)
response.answer

2026/05/05 06:41:49 WARNING dspy.primitives.python_interpreter: Unable to find the Deno cache dir.


Prediction(
    trajectory={'thought_0': 'The user is asking for the number of possible combinations when selecting 9 balls out of 25. This is a combination problem and can be solved using the combination formula: C(n, k) = n! / (k! * (n-k)!), where n is the total number of items and k is the number of items to choose. In this case, n=25 and k=9. I should use the `evaluate_math` tool to calculate this.', 'tool_name_0': 'evaluate_math', 'tool_args_0': {'expr': '25! / (9! * (25-9)!)'}, 'observation_0': 'Execution error in evaluate_math: \nTraceback (most recent call last):\n  File "/Users/aurobindotripathy/prompt-opt-cookbook/.venv/lib/python3.11/site-packages/dspy/primitives/python_interpreter.py", line 322, in _ensure_deno_process\n    self.deno_process = subprocess.Popen(\n                        ^^^^^^^^^^^^^^^^^\n  File "/Users/aurobindotripathy/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/subprocess.py", line 1026, in __init__\n    self._execute_child(ar

0

In [15]:
response.trajectory

{'thought_0': 'The user is asking for the number of possible combinations when selecting 9 balls out of 25. This is a combination problem and can be solved using the combination formula: C(n, k) = n! / (k! * (n-k)!), where n is the total number of items and k is the number of items to choose. In this case, n=25 and k=9. I should use the `evaluate_math` tool to calculate this.',
 'tool_name_0': 'evaluate_math',
 'tool_args_0': {'expr': '25! / (9! * (25-9)!)'},
 'observation_0': 'Execution error in evaluate_math: \nTraceback (most recent call last):\n  File "/Users/aurobindotripathy/prompt-opt-cookbook/.venv/lib/python3.11/site-packages/dspy/primitives/python_interpreter.py", line 322, in _ensure_deno_process\n    self.deno_process = subprocess.Popen(\n                        ^^^^^^^^^^^^^^^^^\n  File "/Users/aurobindotripathy/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/subprocess.py", line 1026, in __init__\n    self._execute_child(args, executable, preexec_

# NPS


In [16]:
import json
with open('nps_comments.json', 'r') as f:
    nps_data = json.loads(f.read())

In [17]:
topics = set()

for rec in nps_data: 
    for t in rec['topics']: 
        topics.add(t)

In [18]:
print(wrap_text(str(list(topics))))

['Inaccurate Product Descriptions or Photos', 'Complicated Returns or
Exchanges', 'Customs and Import Charges', 'Slow or Unreliable Shipping',
'Difficult Product Discovery', 'Limited Size or Shade Availability', 'Damaged or
Incorrect Items', 'Website or App Bugs', 'Unresponsive or Generic Customer
Support', 'Confusing Loyalty or Discount Systems']


In [19]:
from typing import Literal, List

class NPSTopic(dspy.Signature):
    """Classify NPS topics"""

    comment: str = dspy.InputField()
    answer: List[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 
                    'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 
                    'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 
                    'Damaged or Incorrect Items']] = dspy.OutputField()

In [20]:
print(nps_data[0])
print(nps_data[0]['topics'])
print(wrap_text(nps_data[0]['comment'], width=72))

{'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}
['Limited Size or Shade Availability']
Absolutely frustrated! Every time I find something I love, it's sold out
in my size. What's the point of having a wishlist if nothing is ever
available?


In [21]:
nps_topic_model = dspy.ChainOfThought(NPSTopic)
response = nps_topic_model(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?")
print(response)


Prediction(
    reasoning='The user is expressing frustration because the items they are interested in are frequently unavailable in their size, indicating a problem with the range of sizes offered.',
    answer=['Limited Size or Shade Availability']
)


In [22]:
dspy.inspect_history(n = 1)






[2026-05-05T06:41:59.134246]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## comment ## ]]
{comment}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"Slow or Unreliable Shipping\", \"Inaccurate Product Descrip

In [23]:
nps_df = pd.DataFrame(nps_data)
# Net effect: each row gets a 1-based id (1, 2, 3, …, N) in a new column.
nps_df['id'] = list(map(lambda x: x + 1, range(nps_df.shape[0])))

In [24]:
tmp = []

for rec in tqdm.tqdm(nps_df.to_dict('records')):
    response = nps_topic_model(comment = rec['comment'])
    res = {
        'id': rec['id'],
        'model_topics': response.answer
    }

    tmp.append(res)

100%|██████████| 105/105 [10:54<00:00,  6.23s/it]


In [25]:
ini_model_topics_df = pd.DataFrame(tmp)

In [26]:
ini_model_topics_df

,id,model_topics
0,1,[Limited Size or Shade Availability]
1,2,[Difficult Product Discovery]
2,3,[Inaccurate Product Descriptions or Photos]
3,4,"[Confusing Loyalty or Discount Systems, Unresp..."
4,5,[Website or App Bugs]
...,...,...
100,101,[Customs and Import Charges]
101,102,"[Confusing Loyalty or Discount Systems, Compli..."
102,103,"[Limited Size or Shade Availability, Unrespons..."
103,104,"[Difficult Product Discovery, Inaccurate Produ..."


In [27]:
nps_df = nps_df.merge(ini_model_topics_df)

In [28]:
# show head
nps_df.sample(5).to_dict('records')

[{'topics': ['Complicated Returns or Exchanges'],
  'comment': 'Need original packaging to return but your packaging is designed to be opened destructively. Catch-22 situation.',
  'id': 68,
  'model_topics': ['Complicated Returns or Exchanges']},
 {'topics': ['Slow or Unreliable Shipping'],
  'comment': 'Tracking information is never accurate. Package shows as delivered but arrives 3 days later.',
  'id': 66,
  'model_topics': ['Slow or Unreliable Shipping']},
 {'topics': ['Complicated Returns or Exchanges'],
  'comment': 'Have to pay return shipping for defective products. This should be covered by the company, not the customer.',
  'id': 40,
  'model_topics': ['Complicated Returns or Exchanges']},
 {'topics': ['Customs and Import Charges'],
  'comment': 'Charged duties on a return shipment. Having to pay to send back your defective product is insulting.',
  'id': 74,
  'model_topics': ['Customs and Import Charges',
   'Complicated Returns or Exchanges']},
 {'topics': ['Website or Ap

In [29]:
def compare_topics(l1, l2):
    l1_fmt = ', '.join(sorted(l1))
    l2_fmt = ', '.join(sorted(l2))
    if l1_fmt == l2_fmt: 
        return 1 
    return 0


nps_df['model_accuracy'] = list(map(
    compare_topics,
    nps_df.topics,
    nps_df.model_topics
))

In [30]:
round(100*nps_df.model_accuracy.mean(), 2)

np.float64(85.71)

In [31]:
import random
random.random()

0.7571529592889086

In [32]:
trainset = []
valset = []
for rec in nps_data: 
    if random.random() <= 0.5:
        trainset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )
    else: 
        valset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )

In [33]:
# tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)

In [34]:
def list_exact_match(example, pred, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        pred_answer = pred.answer
        expected_answer = example.answer
        
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [35]:
tp = dspy.MIPROv2(metric=list_exact_match, auto="light", num_threads=24)

In [36]:
opt_nps_topic_model =  tp.compile(
    nps_topic_model, 
    trainset=trainset, 
    valset=valset,
    requires_permission_to_run = False, provide_traceback=True)

2026/05/05 06:52:53 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/05/05 06:52:53 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 46

2026/05/05 06:52:53 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/05/05 06:52:53 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/05/05 06:52:53 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


  8%|▊         | 5/59 [00:23<04:12,  4.68s/it]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/6


  5%|▌         | 3/59 [00:33<10:16, 11.01s/it]


...

2026/05/05 07:01:27 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 40.00 / 46 (87.0%): 100%|██████████| 46/46 [00:42<00:00,  1.08it/s]

2026/05/05 07:02:01 INFO dspy.evaluate.evaluate: Average Metric: 40 / 46 (87.0%)
2026/05/05 07:02:01 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 86.96 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 5'].
2026/05/05 07:02:01 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [89.13, 82.61, 80.43, 82.61, 86.96, 84.78, 84.78, 86.96, 82.61, 86.96]
2026/05/05 07:02:01 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 89.13
2026/05/05 07:02:01 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/05/05 07:02:01 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====



Average Metric: 40.00 / 46 (87.0%): 100%|██████████| 46/46 [00:19<00:00,  2.38it/s]

2026/05/05 07:02:21 INFO dspy.evaluate.evaluate: Average Metric: 40 / 46 (87.0%)
2026/05/05 07:02:21 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 86.96 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0'].
2026/05/05 07:02:21 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [89.13, 82.61, 80.43, 82.61, 86.96, 84.78, 84.78, 86.96, 82.61, 86.96, 86.96]
2026/05/05 07:02:21 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 89.13
2026/05/05 07:02:21 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/05/05 07:02:21 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 89.13!


In [37]:
opt_nps_topic_model(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The user is expressing frustration because items they are interested in are frequently out of stock in their size. This directly relates to the availability of products in specific sizes.',
    answer=['Limited Size or Shade Availability']
)

In [38]:
dspy.inspect_history(n = 1)





[2026-05-05T07:02:28.177776]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## comment ## ]]
{comment}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"Slow or Unreliable Shipping\", \"Inaccurate Product Descrip

In [39]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    prev_resp = nps_topic_model(comment = comment) 
    new_resp = opt_nps_topic_model(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'sot_answer': e.answer,
        'prev_answer': prev_resp.answer,
        'new_answer': new_resp.answer
        }
    )

100%|██████████| 46/46 [11:22<00:00, 14.83s/it]


In [40]:
cmp_df = pd.DataFrame(tmp)

In [41]:
def list_exact_match_raw(expected_answer, pred_answer, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [42]:
cmp_df['prev_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.sot_answer,
    cmp_df.prev_answer))

cmp_df['new_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.sot_answer,
    cmp_df.new_answer))

In [43]:
cmp_df[['prev_accuracy', 'new_accuracy']].mean()*100

prev_accuracy    82.608696
new_accuracy     84.782609
dtype: float64

### dspy.BootstrapFewShotWithRandomSearch

In [44]:
tp2 = dspy.BootstrapFewShotWithRandomSearch(list_exact_match, num_threads=24, max_bootstrapped_demos = 10)

Going to sample between 1 and 10 traces per predictor.
Will attempt to bootstrap 16 candidate sets.


In [45]:
opt2_nps_topic_model =  tp2.compile(
    nps_topic_model, 
    trainset=trainset, 
    valset=valset)

Average Metric: 39.00 / 46 (84.8%): 100%|██████████| 46/46 [00:27<00:00,  1.67it/s]

2026/05/05 07:14:18 INFO dspy.evaluate.evaluate: Average Metric: 39 / 46 (84.8%)



New best score: 84.78 for seed -3
Scores so far: [84.78]
Best score so far: 84.78
Average Metric: 40.00 / 46 (87.0%): 100%|██████████| 46/46 [00:25<00:00,  1.79it/s]

2026/05/05 07:14:43 INFO dspy.evaluate.evaluate: Average Metric: 40 / 46 (87.0%)



New best score: 86.96 for seed -2
Scores so far: [84.78, 86.96]
Best score so far: 86.96


...

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Average Metric: 26.00 / 29 (89.7%):  63%|██████▎   | 29/46 [00:12<00:07,  2.26it/s] 

2026/05/05 07:38:15 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 38.00 / 46 (82.6%): 100%|██████████| 46/46 [00:39<00:00,  1.15it/s]

2026/05/05 07:38:41 INFO dspy.evaluate.evaluate: Average Metric: 38 / 46 (82.6%)



Scores so far: [84.78, 86.96, 86.96, 89.13, 84.78, 89.13, 86.96, 84.78, 89.13, 82.61, 84.78, 86.96, 82.61, 86.96, 91.3, 86.96, 84.78, 84.78, 82.61]
Best score so far: 91.3
19 candidate programs found.


In [48]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    prev_resp = nps_topic_model(comment = comment) 
    new_resp = opt_nps_topic_model(comment = comment)
    new_reason_resp = opt2_nps_topic_model(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'sot_answer': e.answer,
        'prev_answer': prev_resp.answer,
        'new_answer': new_resp.answer,
        'new_reason_answer': new_reason_resp.answer
        }
    )

100%|██████████| 46/46 [03:52<00:00,  5.05s/it]


In [50]:
cmp_df = pd.DataFrame(tmp)

In [52]:
cmp_df['prev_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.sot_answer,
    cmp_df.prev_answer))

cmp_df['new_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.sot_answer,
    cmp_df.new_answer))

cmp_df['new_reason_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.sot_answer,
    cmp_df.new_reason_answer))

In [53]:
cmp_df[['prev_accuracy', 'new_accuracy', 'new_reason_accuracy']].mean()*100

prev_accuracy          84.782609
new_accuracy           80.434783
new_reason_accuracy    91.304348
dtype: float64

In [54]:
opt2_nps_topic_model(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The user is expressing significant frustration because the items they are interested in are consistently unavailable in their size. This directly relates to the limited availability of specific product sizes.',
    answer=['Limited Size or Shade Availability']
)

In [55]:
dspy.inspect_history(n = 1)





[2026-05-06T22:59:46.536713]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## comment ## ]]
{comment}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"Slow or Unreliable Shipping\", \"Inaccurate Product Descrip